## Defualt params:
`(500, 500) and (1000, 1000) network area are *NOT* tested for node placement.`
1. Initial energy= 0.5J
2. Comm range= 50m
3. Sink speed= 25m/round
4. Net area= (100, 100)
5. Nodes num= 100

## Basic Methods

In [ ]:
import sys
sys.path.append(r"D:\Papers\4) Finished Articles\6. MWSN - DCHPC\DCHPC\ModelClasses")
from simulation import Simulation
from ConfigClass.config import plot_comparison, visualize_deployment

In [ ]:
def plot_time_series(time_series_dict, label, title, color='b'):
    rounds_list = time_series_dict['rounds']
    values_list = time_series_dict[label]

    min_len = min(len(r) for r in rounds_list)
    
    round_grid = np.full((len(rounds_list), min_len), np.nan)
    value_grid = np.full((len(values_list), min_len), np.nan)

    for i, (rds, vals) in enumerate(zip(rounds_list, values_list)):
        round_grid[i, :] = rds[:min_len]
        value_grid[i, :] = vals[:min_len]

    avg_rounds = np.nanmean(round_grid, axis=0)  
    avg_vals = np.nanmean(value_grid, axis=0)
    std_vals = np.nanstd(value_grid, axis=0)


    valid = ~np.isnan(avg_vals)
    avg_rounds = avg_rounds[valid]
    avg_vals = avg_vals[valid]
    std_vals = std_vals[valid]

    plt.plot(avg_rounds, avg_vals, label=f'{title} (Mean)', color=color, linewidth=2)
    plt.fill_between(avg_rounds, 
                     avg_vals - std_vals, 
                     avg_vals + std_vals, 
                     color=color, alpha=0.2, label=f'{title} ± Std')

In [ ]:
import numpy as np
import random


def run_multiple_simulations(area_size, n_nodes,
                             sink_mode, routing_mode, mode,
                             n_runs=30, CHS="optimizer",
                             edge_threshold=0.4, tune_edge_iterations=20,
                             include_ack_energy=False,
                             num_sinks=1,
                             enable_heterogeneity=False, hetero_mode='two_tier',
                             variable_packet_size=False):

    FND, HND, LND, PDR = [], [], [], []
    TotalGenerated, TotalDelivered = [], []
    Avg_E2E_Delay_Rounds, Avg_E2E_Delay_Sec = [], []

    EC, avg_RE, TH, TH_pps, PLR, LB, FI, CE, Buffer_Overflow_Rate = [
    ], [], [], [], [], [], [], [], []
    CA, RL, EE, EE_Js, routing_Overhead_Bytes, Traffic_Load_Pct, Overhead_Normalized = [
    ], [], [], [], [], [], []

    round_runs, EC_runs, avg_RE_runs = [], [], []
    TH_runs, TH_pps_runs, PLR_runs, LB_runs = [], [], [], []
    FI_runs, CE_runs, Buffer_Overflow_Rate_runs = [], [], []
    CA_runs, RL_runs, EE_runs, EE_Js_runs = [], [], [], []
    routing_Overhead_Bytes_runs, Traffic_Load_Pct_runs, Overhead_Normalized_runs = [], [], []

    for seed in range(n_runs):
        np.random.seed(seed)
        random.seed(seed)
        sim = Simulation(
            area_size=area_size,
            n_nodes=n_nodes,
            rounds=60000,
            init_energy=0.5,
            comm_range=50.0,
            sink_mode=sink_mode,
            routing_mode=routing_mode,
            seed=seed,
            localization_mode=mode,
            head_selection_strategy=CHS,

            edge_threshold=edge_threshold,
            tune_edge_iterations=tune_edge_iterations,

            include_ack_energy=include_ack_energy,
            num_sinks=num_sinks,

            enable_heterogeneity=enable_heterogeneity,
            hetero_mode=hetero_mode,
            variable_packet_size=variable_packet_size
        )
        metrics, detailed_metrics = sim.run()

        rounds = np.array(detailed_metrics['round'])
        EC_vals = np.array(detailed_metrics['EC'])
        avg_RE = np.array(detailed_metrics['avg_RE'])
        TH = np.array(detailed_metrics['TH'])
        TH_pps = np.array(detailed_metrics['TH_pps'])
        PLR = np.array(detailed_metrics['PLR'])
        LB = np.array(detailed_metrics['LB'])
        FI = np.array(detailed_metrics['FI'])
        CE = np.array(detailed_metrics['CE'])
        Buffer_Overflow_Rate = np.array(
            detailed_metrics['Buffer_Overflow_Rate'])
        CA = np.array(detailed_metrics['CA'])
        RL = np.array(detailed_metrics['RL'])
        EE = np.array(detailed_metrics['EE'])
        EE_Js = np.array(detailed_metrics['EE_Js'])
        routing_Overhead_Bytes = np.array(
            detailed_metrics['Routing_Overhead_Bytes'])
        Traffic_Load_Pct = np.array(detailed_metrics['Traffic_Load_Pct'])
        Overhead_Normalized = np.array(detailed_metrics['Overhead_Normalized'])

        round_runs.append(rounds)
        EC_runs.append(EC_vals)
        avg_RE_runs.append(avg_RE)
        TH_runs.append(TH)
        TH_pps_runs.append(TH_pps)
        PLR_runs.append(PLR)
        LB_runs.append(LB)
        FI_runs.append(FI)
        CE_runs.append(CE)
        Buffer_Overflow_Rate_runs.append(
            Buffer_Overflow_Rate)
        CA_runs.append(CA)
        RL_runs.append(RL)
        EE_runs.append(EE)
        EE_Js_runs.append(EE_Js)
        routing_Overhead_Bytes_runs.append(
            routing_Overhead_Bytes)
        Traffic_Load_Pct_runs.append(Traffic_Load_Pct)
        Overhead_Normalized_runs.append(Overhead_Normalized)

        plot_res = {
            'rounds': round_runs,
            'EC': EC_runs,
            'avg_RE': avg_RE_runs,
            'TH': TH_runs,
            'TH_pps': TH_pps_runs,
            'PLR': PLR_runs,
            'LB': LB_runs,
            'FI': FI_runs,
            'CE': CE_runs,
            'Buffer_Overflow_Rate': Buffer_Overflow_Rate_runs,
            'CA': CA_runs,
            'RL': RL_runs,
            'EE': EE_runs,
            'EE_Js': EE_Js_runs,
            'routing_Overhead_Bytes': routing_Overhead_Bytes_runs,
            'Traffic_Load_Pct': Traffic_Load_Pct_runs,
            'Overhead_Normalized': Overhead_Normalized_runs

        }

        FND.append(metrics['FND'])
        HND.append(metrics['HND'])
        LND.append(metrics['LND'])
        PDR.append(metrics['PDR'])
        TotalGenerated.append(metrics['TotalGenerated'])
        TotalDelivered.append(metrics['TotalDelivered'])
        Avg_E2E_Delay_Rounds.append(metrics['Avg_E2E_Delay_Rounds'])
        Avg_E2E_Delay_Sec.append(metrics['Avg_E2E_Delay_Sec'])
        print(f"Seed = {seed}, Mode = {mode} results:")
        print(
            f"FND: {metrics['FND']}, HND: {metrics['HND']}, LND: {metrics['LND']}")
        print(
            f"PDR: {metrics['PDR']}, Total Generated: {metrics['TotalGenerated']}, Total Delivered: {metrics['TotalDelivered']}")
        print(
            f"Avg E2E Delay Rounds: {metrics['Avg_E2E_Delay_Rounds']}, Avg E2E Delay Sec: {metrics['Avg_E2E_Delay_Sec']}")
        print("\n")

    return [np.array(FND), np.array(HND), np.array(LND),
            np.array(PDR), np.array(TotalGenerated), np.array(TotalDelivered),
            np.array(Avg_E2E_Delay_Rounds), np.array(Avg_E2E_Delay_Sec)], plot_res

## Random Node Deployment vs. Voronoi-RL 

### (100, 100) area size, 100 nodes

In [ ]:
# Run both methods
drl_lifetimes, plot_drl = run_multiple_simulations(area_size=(
    100, 100), n_nodes=100, sink_mode="eeosp",
    routing_mode="multi-hop", mode="random", n_runs=30, tune_edge_iterations=50)

baseline_lifetimes, plot_base = run_multiple_simulations(area_size=(
    100, 100), n_nodes=100, sink_mode="eeosp", routing_mode="multi-hop", mode="random", n_runs=30)


print(
    f"Mean DRL FND: {drl_lifetimes[0].mean():.2f} ± {drl_lifetimes[0].std():.2f}")
print(
    f"Mean DRL HND: {drl_lifetimes[1].mean():.2f} ± {drl_lifetimes[1].std():.2f}")
print(
    f"Mean DRL LND: {drl_lifetimes[2].mean():.2f} ± {drl_lifetimes[2].std():.2f}")
print(
    f"Mean DRL PDR: {drl_lifetimes[3].mean():.2f} ± {drl_lifetimes[3].std():.2f}")
print(
    f"Mean DRL TotalGenerated: {drl_lifetimes[4].mean():.2f} ± {drl_lifetimes[4].std():.2f}")
print(
    f"Mean DRL TotalDelivered: {drl_lifetimes[5].mean():.2f} ± {drl_lifetimes[5].std():.2f}")
print(
    f"Mean DRL Avg_E2E_Delay_Rounds: {drl_lifetimes[6].mean():.2f} ± {drl_lifetimes[6].std():.2f}")
print(
    f"Mean DRL Avg_E2E_Delay_Sec: {drl_lifetimes[7].mean():.2f} ± {drl_lifetimes[7].std():.2f}")


print("\n\n\n")
print(
    f"Mean Baseline FND: {baseline_lifetimes[0].mean():.2f} ± {baseline_lifetimes[0].std():.2f}")
print(
    f"Mean Baseline HND: {baseline_lifetimes[1].mean():.2f} ± {baseline_lifetimes[1].std():.2f}")
print(
    f"Mean Baseline LND: {baseline_lifetimes[2].mean():.2f} ± {baseline_lifetimes[2].std():.2f}")
print(
    f"Mean Baseline PDR: {baseline_lifetimes[3].mean():.2f} ± {baseline_lifetimes[3].std():.2f}")
print(
    f"Mean Baseline TotalGenerated: {baseline_lifetimes[4].mean():.2f} ± {baseline_lifetimes[4].std():.2f}")
print(
    f"Mean Baseline TotalDelivered: {baseline_lifetimes[5].mean():.2f} ± {baseline_lifetimes[5].std():.2f}")
print(
    f"Mean Baseline Avg_E2E_Delay_Rounds: {baseline_lifetimes[6].mean():.2f} ± {baseline_lifetimes[6].std():.2f}")
print(
    f"Mean Baseline Avg_E2E_Delay_Sec: {baseline_lifetimes[7].mean():.2f} ± {baseline_lifetimes[7].std():.2f}")

```python
Edge nodes: [3, 4, 7, 8, 9, 10, 13, 17, 19, 23, 26, 27, 30, 33, 34, 37, 38, 41, 44, 46, 48, 49, 51, 54, 59, 61, 63, 66, 71, 73, 74, 75, 77, 79, 82, 83, 85, 86, 88, 89, 96]
RL Optimizing 41 edge nodes using Q-Learning...
  Iter 0: Epsilon=0.85, Global Reward=0.9518
  Iter 20: Epsilon=0.31, Global Reward=0.9515
  Iter 40: Epsilon=0.11, Global Reward=0.9515
Sink speed: 25.0 m/round = 0.50 m/s
Seed = 0, Mode = DRL results:
FND: 159, HND: 245, LND: 11995
PDR: 0.9333689268553124, Total Generated: 18731, Total Delivered: 17482
Avg E2E Delay Rounds: 0.06069099645349502, Avg E2E Delay Sec: 3.034549822674751


Edge nodes: [1, 2, 6, 10, 12, 13, 14, 16, 19, 20, 23, 25, 27, 28, 34, 35, 38, 43, 45, 48, 51, 52, 53, 54, 55, 56, 58, 59, 60, 61, 65, 66, 67, 69, 70, 71, 72, 74, 75, 79, 82, 85, 91, 95, 97, 98]
RL Optimizing 46 edge nodes using Q-Learning...
  Iter 0: Epsilon=0.85, Global Reward=0.9523
  Iter 20: Epsilon=0.31, Global Reward=0.9520
  Iter 40: Epsilon=0.11, Global Reward=0.9521
Sink speed: 25.0 m/round = 0.50 m/s
Seed = 1, Mode = DRL results:
FND: 142, HND: 216, LND: 12035
PDR: 0.9321107865411663, Total Generated: 18487, Total Delivered: 17231
Avg E2E Delay Rounds: 0.06337415123904591, Avg E2E Delay Sec: 3.1687075619522957


Edge nodes: [7, 9, 12, 19, 20, 21, 24, 29, 34, 37, 44, 45, 49, 51, 53, 55, 63, 68, 69, 71, 73, 75, 77, 78, 82, 86, 89, 91, 94, 95, 98, 99]
RL Optimizing 32 edge nodes using Q-Learning...
  Iter 0: Epsilon=0.85, Global Reward=0.9517
  Iter 20: Epsilon=0.31, Global Reward=0.9517
  Iter 40: Epsilon=0.11, Global Reward=0.9521
Sink speed: 25.0 m/round = 0.50 m/s
Seed = 2, Mode = DRL results:
FND: 157, HND: 226, LND: 7273
PDR: 0.912476330738481, Total Generated: 19013, Total Delivered: 17348
Avg E2E Delay Rounds: 0.06265851971408808, Avg E2E Delay Sec: 3.1329259857044036


Edge nodes: [2, 3, 4, 5, 8, 14, 15, 16, 33, 34, 37, 38, 39, 41, 44, 47, 50, 51, 56, 57, 64, 65, 68, 69, 70, 74, 76, 77, 78, 80, 82, 83, 84, 90, 91, 93, 97]
RL Optimizing 37 edge nodes using Q-Learning...
  Iter 0: Epsilon=0.85, Global Reward=0.9518
  Iter 20: Epsilon=0.31, Global Reward=0.9518
  Iter 40: Epsilon=0.11, Global Reward=0.9521
Sink speed: 25.0 m/round = 0.50 m/s
Seed = 3, Mode = DRL results:
FND: 191, HND: 219, LND: 19230
PDR: 0.9466518917207535, Total Generated: 18952, Total Delivered: 17940
Avg E2E Delay Rounds: 0.05925306577480491, Avg E2E Delay Sec: 2.9626532887402455


Edge nodes: [0, 1, 2, 3, 6, 8, 9, 10, 12, 16, 21, 32, 33, 37, 40, 45, 52, 55, 58, 59, 61, 62, 63, 64, 68, 74, 75, 77, 82, 83, 84, 87, 90, 91, 92, 96, 99]
RL Optimizing 37 edge nodes using Q-Learning...
  Iter 0: Epsilon=0.85, Global Reward=0.9522
  Iter 20: Epsilon=0.31, Global Reward=0.9520
  Iter 40: Epsilon=0.11, Global Reward=0.9521
Sink speed: 25.0 m/round = 0.50 m/s
Seed = 4, Mode = DRL results:
FND: 147, HND: 228, LND: 12141
PDR: 0.9248935949340807, Total Generated: 19267, Total Delivered: 17819
Avg E2E Delay Rounds: 0.06493069195802234, Avg E2E Delay Sec: 3.246534597901117


Edge nodes: [0, 1, 5, 6, 7, 13, 14, 15, 17, 18, 19, 21, 22, 23, 26, 27, 28, 31, 32, 33, 35, 39, 40, 43, 44, 45, 48, 50, 51, 54, 56, 58, 67, 68, 69, 72, 75, 79, 82, 85, 93, 99]
RL Optimizing 42 edge nodes using Q-Learning...
  Iter 0: Epsilon=0.85, Global Reward=0.9520
  Iter 20: Epsilon=0.31, Global Reward=0.9519
  Iter 40: Epsilon=0.11, Global Reward=0.9519
Sink speed: 25.0 m/round = 0.50 m/s
Seed = 5, Mode = DRL results:
FND: 169, HND: 208, LND: 12646
PDR: 0.9381816127027179, Total Generated: 17698, Total Delivered: 16603
Avg E2E Delay Rounds: 0.06571101608143107, Avg E2E Delay Sec: 3.2855508040715535


Edge nodes: [0, 1, 7, 9, 10, 11, 13, 15, 18, 21, 23, 30, 31, 34, 36, 37, 40, 41, 43, 47, 50, 53, 54, 56, 58, 64, 66, 71, 72, 73, 74, 76, 78, 79, 81, 82, 83, 84, 85, 86, 90, 91, 93, 97, 98]
RL Optimizing 45 edge nodes using Q-Learning...
  Iter 0: Epsilon=0.85, Global Reward=0.9521
  Iter 20: Epsilon=0.31, Global Reward=0.9520
  Iter 40: Epsilon=0.11, Global Reward=0.9519
Sink speed: 25.0 m/round = 0.50 m/s
Seed = 6, Mode = DRL results:
FND: 163, HND: 226, LND: 4351
PDR: 0.9237547686851861, Total Generated: 18612, Total Delivered: 17192
Avg E2E Delay Rounds: 0.06456491391344811, Avg E2E Delay Sec: 3.2282456956724057


Edge nodes: [0, 2, 3, 7, 9, 10, 12, 23, 27, 28, 30, 33, 35, 45, 46, 50, 53, 57, 62, 64, 65, 66, 68, 71, 72, 73, 74, 75, 79, 80, 81, 82, 83, 84, 85, 86, 87, 94, 98]
RL Optimizing 39 edge nodes using Q-Learning...
  Iter 0: Epsilon=0.85, Global Reward=0.9520
  Iter 20: Epsilon=0.31, Global Reward=0.9519
  Iter 40: Epsilon=0.11, Global Reward=0.9521
Sink speed: 25.0 m/round = 0.50 m/s
Seed = 7, Mode = DRL results:
FND: 134, HND: 217, LND: 3046
PDR: 0.9436651583710407, Total Generated: 17681, Total Delivered: 16684
Avg E2E Delay Rounds: 0.06868856389355071, Avg E2E Delay Sec: 3.434428194677536


Edge nodes: [0, 2, 8, 10, 12, 14, 15, 17, 20, 21, 23, 24, 28, 29, 32, 40, 49, 50, 51, 54, 56, 57, 61, 63, 65, 68, 71, 76, 82, 85, 88, 89, 91, 92, 95, 96, 99]
RL Optimizing 37 edge nodes using Q-Learning...
  Iter 0: Epsilon=0.85, Global Reward=0.9517
  Iter 20: Epsilon=0.31, Global Reward=0.9513
  Iter 40: Epsilon=0.11, Global Reward=0.9517
Sink speed: 25.0 m/round = 0.50 m/s
Seed = 8, Mode = DRL results:
FND: 164, HND: 249, LND: 34590
PDR: 0.9426419155232715, Total Generated: 18586, Total Delivered: 17519
Avg E2E Delay Rounds: 0.07112278098064959, Avg E2E Delay Sec: 3.5561390490324793


Edge nodes: [0, 2, 4, 5, 6, 12, 13, 14, 16, 17, 18, 24, 26, 27, 32, 36, 41, 42, 44, 46, 47, 50, 51, 52, 56, 60, 62, 63, 64, 73, 74, 77, 80, 81, 84, 86, 94, 99]
RL Optimizing 38 edge nodes using Q-Learning...
  Iter 0: Epsilon=0.85, Global Reward=0.9517
  Iter 20: Epsilon=0.31, Global Reward=0.9514
  Iter 40: Epsilon=0.11, Global Reward=0.9514
Sink speed: 25.0 m/round = 0.50 m/s
Seed = 9, Mode = DRL results:
FND: 168, HND: 221, LND: 22510
PDR: 0.9328545371637027, Total Generated: 17545, Total Delivered: 16366
Avg E2E Delay Rounds: 0.049126237321275815, Avg E2E Delay Sec: 2.456311866063791


Edge nodes: [0, 4, 5, 6, 9, 16, 18, 19, 26, 27, 28, 29, 30, 32, 37, 38, 42, 46, 48, 53, 56, 57, 58, 64, 67, 69, 74, 77, 80, 85, 87, 88, 91, 92, 96]
RL Optimizing 35 edge nodes using Q-Learning...
  Iter 0: Epsilon=0.85, Global Reward=0.9520
  Iter 20: Epsilon=0.31, Global Reward=0.9518
  Iter 40: Epsilon=0.11, Global Reward=0.9517
Sink speed: 25.0 m/round = 0.50 m/s
Seed = 10, Mode = DRL results:
FND: 191, HND: 208, LND: 18283
PDR: 0.9260649087221096, Total Generated: 19721, Total Delivered: 18262
Avg E2E Delay Rounds: 0.09193954659949623, Avg E2E Delay Sec: 4.596977329974812


Edge nodes: [0, 3, 4, 5, 6, 8, 9, 12, 18, 21, 22, 23, 27, 28, 30, 34, 36, 38, 39, 52, 53, 55, 64, 65, 67, 68, 74, 75, 76, 78, 80, 81, 82, 85, 87, 89, 90, 92, 93, 99]
RL Optimizing 40 edge nodes using Q-Learning...
  Iter 0: Epsilon=0.85, Global Reward=0.9520
  Iter 20: Epsilon=0.31, Global Reward=0.9520
  Iter 40: Epsilon=0.11, Global Reward=0.9519
Sink speed: 25.0 m/round = 0.50 m/s
Seed = 11, Mode = DRL results:
FND: 139, HND: 217, LND: 4196
PDR: 0.945777677156441, Total Generated: 17669, Total Delivered: 16710
Avg E2E Delay Rounds: 0.05918611609814482, Avg E2E Delay Sec: 2.959305804907241


Edge nodes: [2, 3, 4, 6, 7, 9, 10, 11, 16, 21, 24, 26, 27, 31, 45, 46, 47, 48, 49, 52, 53, 54, 56, 59, 61, 62, 65, 67, 68, 69, 70, 71, 82, 87, 89, 90, 91, 92, 93, 99]
RL Optimizing 40 edge nodes using Q-Learning...
  Iter 0: Epsilon=0.85, Global Reward=0.9519
  Iter 20: Epsilon=0.31, Global Reward=0.9518
  Iter 40: Epsilon=0.11, Global Reward=0.9516
Sink speed: 25.0 m/round = 0.50 m/s
Seed = 12, Mode = DRL results:
FND: 150, HND: 217, LND: 14243
PDR: 0.9532568342895714, Total Generated: 17779, Total Delivered: 16947
Avg E2E Delay Rounds: 0.06945182038118841, Avg E2E Delay Sec: 3.4725910190594207


Edge nodes: [1, 2, 5, 6, 9, 10, 11, 12, 13, 14, 15, 17, 18, 21, 24, 26, 28, 32, 34, 37, 41, 44, 45, 46, 47, 48, 50, 52, 54, 60, 62, 63, 64, 66, 68, 71, 79, 81, 82, 83, 84, 86, 87, 91, 93, 96, 97, 98]
RL Optimizing 48 edge nodes using Q-Learning...
  Iter 0: Epsilon=0.85, Global Reward=0.9522
  Iter 20: Epsilon=0.31, Global Reward=0.9518
  Iter 40: Epsilon=0.11, Global Reward=0.9520
Sink speed: 25.0 m/round = 0.50 m/s
Seed = 13, Mode = DRL results:
FND: 126, HND: 240, LND: 12766
PDR: 0.8818913086129095, Total Generated: 20389, Total Delivered: 17980
Avg E2E Delay Rounds: 0.0, Avg E2E Delay Sec: 0.0


Edge nodes: [1, 2, 9, 15, 16, 18, 22, 24, 26, 27, 32, 39, 40, 41, 43, 45, 49, 51, 54, 55, 59, 60, 64, 66, 77, 78, 80, 91, 92, 97, 99]
RL Optimizing 31 edge nodes using Q-Learning...
  Iter 0: Epsilon=0.85, Global Reward=0.9517
  Iter 20: Epsilon=0.31, Global Reward=0.9517
  Iter 40: Epsilon=0.11, Global Reward=0.9517
Sink speed: 25.0 m/round = 0.50 m/s
Seed = 14, Mode = DRL results:
FND: 154, HND: 203, LND: 24737
PDR: 0.9477041549336245, Total Generated: 17402, Total Delivered: 16491
Avg E2E Delay Rounds: 0.06445940209811413, Avg E2E Delay Sec: 3.2229701049057065


Edge nodes: [0, 1, 5, 6, 8, 9, 10, 12, 20, 21, 22, 23, 25, 29, 30, 36, 39, 44, 45, 50, 51, 52, 55, 58, 61, 62, 64, 70, 75, 76, 77, 80, 81, 82, 86, 88, 89, 90, 93, 99]
RL Optimizing 40 edge nodes using Q-Learning...
  Iter 0: Epsilon=0.85, Global Reward=0.9517
  Iter 20: Epsilon=0.31, Global Reward=0.9518
  Iter 40: Epsilon=0.11, Global Reward=0.9519
Sink speed: 25.0 m/round = 0.50 m/s
Seed = 15, Mode = DRL results:
FND: 149, HND: 225, LND: 22801
PDR: 0.9252408904640602, Total Generated: 18059, Total Delivered: 16708
Avg E2E Delay Rounds: 0.0703255925305243, Avg E2E Delay Sec: 3.516279626526215


Edge nodes: [1, 4, 6, 10, 12, 15, 16, 19, 20, 24, 25, 29, 30, 31, 35, 38, 40, 43, 44, 45, 46, 48, 49, 53, 56, 57, 62, 63, 69, 71, 76, 79, 86, 89, 90, 92, 94, 95, 96, 98]
RL Optimizing 40 edge nodes using Q-Learning...
  Iter 0: Epsilon=0.85, Global Reward=0.9520
  Iter 20: Epsilon=0.31, Global Reward=0.9517
  Iter 40: Epsilon=0.11, Global Reward=0.9519
Sink speed: 25.0 m/round = 0.50 m/s
Seed = 16, Mode = DRL results:
FND: 170, HND: 208, LND: 14779
PDR: 0.9294368797599816, Total Generated: 17333, Total Delivered: 16109
Avg E2E Delay Rounds: 0.07610652430318456, Avg E2E Delay Sec: 3.8053262151592278


Edge nodes: [1, 4, 5, 6, 7, 13, 15, 20, 21, 23, 24, 25, 28, 29, 30, 31, 32, 34, 40, 41, 46, 47, 57, 59, 62, 65, 67, 74, 75, 80, 82, 83, 85, 91, 92, 94, 97, 99]
RL Optimizing 38 edge nodes using Q-Learning...
  Iter 0: Epsilon=0.85, Global Reward=0.9521
  Iter 20: Epsilon=0.31, Global Reward=0.9520
  Iter 40: Epsilon=0.11, Global Reward=0.9518
Sink speed: 25.0 m/round = 0.50 m/s
Seed = 17, Mode = DRL results:
FND: 170, HND: 217, LND: 16145
PDR: 0.9385196540019709, Total Generated: 18267, Total Delivered: 17143
Avg E2E Delay Rounds: 0.0705244122965642, Avg E2E Delay Sec: 3.52622061482821


Edge nodes: [1, 3, 4, 6, 7, 12, 20, 22, 23, 24, 27, 34, 35, 36, 39, 42, 44, 46, 57, 60, 62, 67, 71, 72, 76, 78, 82, 83, 84, 88, 91, 94, 98]
RL Optimizing 33 edge nodes using Q-Learning...
  Iter 0: Epsilon=0.85, Global Reward=0.9517
  Iter 20: Epsilon=0.31, Global Reward=0.9517
  Iter 40: Epsilon=0.11, Global Reward=0.9518
Sink speed: 25.0 m/round = 0.50 m/s
Seed = 18, Mode = DRL results:
FND: 151, HND: 266, LND: 18897
PDR: 0.8968607761250059, Total Generated: 21312, Total Delivered: 19113
Avg E2E Delay Rounds: 0.0286192643750327, Avg E2E Delay Sec: 1.430963218751635


Edge nodes: [0, 2, 4, 10, 15, 18, 20, 21, 24, 26, 27, 28, 30, 31, 32, 36, 37, 38, 39, 41, 45, 47, 49, 50, 52, 53, 56, 57, 58, 59, 60, 64, 66, 71, 75, 79, 80, 84, 86, 89, 90, 91, 93, 95]
RL Optimizing 44 edge nodes using Q-Learning...
  Iter 0: Epsilon=0.85, Global Reward=0.9521
  Iter 20: Epsilon=0.31, Global Reward=0.9519
  Iter 40: Epsilon=0.11, Global Reward=0.9518
Sink speed: 25.0 m/round = 0.50 m/s
Seed = 19, Mode = DRL results:
FND: 179, HND: 229, LND: 15124
PDR: 0.9374194933447831, Total Generated: 18633, Total Delivered: 17466
Avg E2E Delay Rounds: 0.05456315126531547, Avg E2E Delay Sec: 2.7281575632657735


Edge nodes: [2, 7, 8, 9, 10, 16, 19, 24, 30, 34, 36, 40, 41, 42, 43, 44, 46, 57, 59, 60, 62, 69, 70, 72, 73, 77, 79, 91, 97]
RL Optimizing 29 edge nodes using Q-Learning...
  Iter 0: Epsilon=0.85, Global Reward=0.9514
  Iter 20: Epsilon=0.31, Global Reward=0.9516
  Iter 40: Epsilon=0.11, Global Reward=0.9516
Sink speed: 25.0 m/round = 0.50 m/s
Seed = 20, Mode = DRL results:
FND: 143, HND: 218, LND: 9237
PDR: 0.9214896281374204, Total Generated: 18368, Total Delivered: 16925
Avg E2E Delay Rounds: 0.07692762186115214, Avg E2E Delay Sec: 3.846381093057607


Edge nodes: [0, 1, 2, 5, 6, 8, 12, 23, 24, 26, 35, 37, 39, 40, 44, 45, 47, 48, 51, 55, 58, 66, 68, 74, 76, 81, 85, 87, 88, 90, 91, 92, 93, 94, 97]
RL Optimizing 35 edge nodes using Q-Learning...
  Iter 0: Epsilon=0.85, Global Reward=0.9521
  Iter 20: Epsilon=0.31, Global Reward=0.9517
  Iter 40: Epsilon=0.11, Global Reward=0.9522
Sink speed: 25.0 m/round = 0.50 m/s
Seed = 21, Mode = DRL results:
FND: 180, HND: 243, LND: 5490
PDR: 0.9227194666127891, Total Generated: 19799, Total Delivered: 18268
Avg E2E Delay Rounds: 0.03164002627545435, Avg E2E Delay Sec: 1.5820013137727174


Edge nodes: [5, 7, 8, 14, 16, 17, 20, 22, 23, 27, 28, 30, 34, 35, 36, 42, 43, 44, 45, 48, 49, 53, 56, 57, 61, 62, 63, 68, 70, 74, 75, 78, 79, 82, 84, 87, 88, 92, 95, 97]
RL Optimizing 40 edge nodes using Q-Learning...
  Iter 0: Epsilon=0.85, Global Reward=0.9519
  Iter 20: Epsilon=0.31, Global Reward=0.9518
  Iter 40: Epsilon=0.11, Global Reward=0.9521
Sink speed: 25.0 m/round = 0.50 m/s
Seed = 22, Mode = DRL results:
FND: 172, HND: 223, LND: 29645
PDR: 0.9358824426704632, Total Generated: 19668, Total Delivered: 18406
Avg E2E Delay Rounds: 0.05677496468542866, Avg E2E Delay Sec: 2.838748234271433


Edge nodes: [0, 5, 7, 8, 11, 12, 17, 24, 26, 29, 30, 31, 34, 37, 38, 39, 41, 46, 47, 48, 52, 54, 55, 57, 58, 63, 67, 68, 70, 71, 73, 76, 77, 81, 82, 84, 85, 91, 92, 95, 96, 97]
RL Optimizing 42 edge nodes using Q-Learning...
  Iter 0: Epsilon=0.85, Global Reward=0.9517
  Iter 20: Epsilon=0.31, Global Reward=0.9518
  Iter 40: Epsilon=0.11, Global Reward=0.9518
Sink speed: 25.0 m/round = 0.50 m/s
Seed = 23, Mode = DRL results:
FND: 155, HND: 237, LND: 7913
PDR: 0.8957509380387385, Total Generated: 19723, Total Delivered: 17666
Avg E2E Delay Rounds: 0.02184988112758972, Avg E2E Delay Sec: 1.092494056379486


Edge nodes: [0, 1, 4, 6, 12, 21, 24, 25, 26, 31, 35, 37, 39, 42, 43, 44, 46, 49, 51, 54, 57, 58, 71, 76, 78, 79, 80, 84, 85, 88, 90, 91, 95, 96, 98]
RL Optimizing 35 edge nodes using Q-Learning...
  Iter 0: Epsilon=0.85, Global Reward=0.9515
  Iter 20: Epsilon=0.31, Global Reward=0.9517
  Iter 40: Epsilon=0.11, Global Reward=0.9516
Sink speed: 25.0 m/round = 0.50 m/s
Seed = 24, Mode = DRL results:
FND: 166, HND: 233, LND: 11278
PDR: 0.92746816948445, Total Generated: 19165, Total Delivered: 17774
Avg E2E Delay Rounds: 0.06031281647350062, Avg E2E Delay Sec: 3.0156408236750307


Edge nodes: [0, 19, 22, 31, 32, 35, 37, 46, 48, 50, 54, 58, 62, 63, 65, 66, 69, 71, 73, 74, 75, 76, 77, 78, 82, 83, 85, 91, 92, 93, 96, 97]
RL Optimizing 32 edge nodes using Q-Learning...
  Iter 0: Epsilon=0.85, Global Reward=0.9515
  Iter 20: Epsilon=0.31, Global Reward=0.9514
  Iter 40: Epsilon=0.11, Global Reward=0.9516
Sink speed: 25.0 m/round = 0.50 m/s
Seed = 25, Mode = DRL results:
FND: 144, HND: 225, LND: 21368
PDR: 0.9384590447374108, Total Generated: 18509, Total Delivered: 17369
Avg E2E Delay Rounds: 0.07087339512925327, Avg E2E Delay Sec: 3.543669756462663


Edge nodes: [2, 8, 11, 15, 22, 23, 24, 25, 27, 30, 32, 34, 35, 36, 37, 40, 44, 45, 46, 48, 49, 50, 53, 57, 62, 63, 64, 66, 69, 70, 72, 74, 75, 76, 77, 78, 79, 80, 81, 82, 85, 87, 88, 92, 95, 96, 99]
RL Optimizing 47 edge nodes using Q-Learning...
  Iter 0: Epsilon=0.85, Global Reward=0.9522
  Iter 20: Epsilon=0.31, Global Reward=0.9521
  Iter 40: Epsilon=0.11, Global Reward=0.9521
Sink speed: 25.0 m/round = 0.50 m/s
Seed = 26, Mode = DRL results:
FND: 177, HND: 213, LND: 15252
PDR: 0.9295355753832075, Total Generated: 17485, Total Delivered: 16252
Avg E2E Delay Rounds: 0.08398966281073099, Avg E2E Delay Sec: 4.199483140536549


Edge nodes: [2, 3, 5, 9, 10, 11, 12, 13, 15, 16, 19, 21, 29, 31, 34, 35, 38, 40, 41, 43, 45, 46, 53, 59, 61, 62, 66, 67, 70, 71, 74, 75, 76, 82, 84, 86, 92, 93, 96, 97, 99]
RL Optimizing 41 edge nodes using Q-Learning...
  Iter 0: Epsilon=0.85, Global Reward=0.9519
  Iter 20: Epsilon=0.31, Global Reward=0.9513
  Iter 40: Epsilon=0.11, Global Reward=0.9514
Sink speed: 25.0 m/round = 0.50 m/s
Seed = 27, Mode = DRL results:
FND: 170, HND: 212, LND: 21923
PDR: 0.9406134686346863, Total Generated: 17345, Total Delivered: 16314
Avg E2E Delay Rounds: 0.05081525070491602, Avg E2E Delay Sec: 2.540762535245801


Edge nodes: [3, 4, 5, 6, 7, 8, 12, 13, 14, 15, 17, 20, 21, 24, 27, 31, 34, 35, 37, 43, 45, 48, 49, 51, 56, 61, 63, 65, 67, 74, 76, 78, 80, 81, 83, 84, 85, 91, 93]
RL Optimizing 39 edge nodes using Q-Learning...
  Iter 0: Epsilon=0.85, Global Reward=0.9521
  Iter 20: Epsilon=0.31, Global Reward=0.9520
  Iter 40: Epsilon=0.11, Global Reward=0.9520
Sink speed: 25.0 m/round = 0.50 m/s
Seed = 28, Mode = DRL results:
FND: 161, HND: 200, LND: 19378
PDR: 0.927029861456091, Total Generated: 18118, Total Delivered: 16795
Avg E2E Delay Rounds: 0.07430782971122357, Avg E2E Delay Sec: 3.7153914855611787


Edge nodes: [1, 6, 9, 10, 11, 17, 21, 22, 23, 25, 27, 29, 33, 37, 39, 43, 46, 50, 57, 62, 66, 67, 74, 75, 77, 78, 79, 83, 84, 85, 87, 88, 89, 91, 92, 93, 96, 97]
RL Optimizing 38 edge nodes using Q-Learning...
  Iter 0: Epsilon=0.85, Global Reward=0.9520
  Iter 20: Epsilon=0.31, Global Reward=0.9518
  Iter 40: Epsilon=0.11, Global Reward=0.9517
Sink speed: 25.0 m/round = 0.50 m/s
Seed = 29, Mode = DRL results:
FND: 157, HND: 205, LND: 9833
PDR: 0.9313011828935396, Total Generated: 17585, Total Delivered: 16376
Avg E2E Delay Rounds: 0.07065217391304347, Avg E2E Delay Sec: 3.532608695652174


Sink speed: 25.0 m/round = 0.50 m/s
Seed = 0, Mode = random results:
FND: 165, HND: 227, LND: 24467
PDR: 0.9295908658420552, Total Generated: 18919, Total Delivered: 17586
Avg E2E Delay Rounds: 0.05242806778118958, Avg E2E Delay Sec: 2.621403389059479


Sink speed: 25.0 m/round = 0.50 m/s
Seed = 1, Mode = random results:
FND: 167, HND: 208, LND: 4909
PDR: 0.934991974317817, Total Generated: 17445, Total Delivered: 16310
Avg E2E Delay Rounds: 0.08546903740036788, Avg E2E Delay Sec: 4.273451870018394


Sink speed: 25.0 m/round = 0.50 m/s
Seed = 2, Mode = random results:
FND: 161, HND: 246, LND: 13056
PDR: 0.8873851378345985, Total Generated: 20025, Total Delivered: 17769
Avg E2E Delay Rounds: 0.03815633969272328, Avg E2E Delay Sec: 1.907816984636164


Sink speed: 25.0 m/round = 0.50 m/s
Seed = 3, Mode = random results:
FND: 170, HND: 219, LND: 10944
PDR: 0.9195214176872284, Total Generated: 17720, Total Delivered: 16293
Avg E2E Delay Rounds: 0.07199410789909777, Avg E2E Delay Sec: 3.5997053949548885


Sink speed: 25.0 m/round = 0.50 m/s
Seed = 4, Mode = random results:
FND: 173, HND: 207, LND: 4024
PDR: 0.9294542581409786, Total Generated: 17720, Total Delivered: 16469
Avg E2E Delay Rounds: 0.06563847228125569, Avg E2E Delay Sec: 3.2819236140627845


Sink speed: 25.0 m/round = 0.50 m/s
Seed = 5, Mode = random results:
FND: 161, HND: 223, LND: 8322
PDR: 0.9279319111307616, Total Generated: 18095, Total Delivered: 16790
Avg E2E Delay Rounds: 0.06468135795116141, Avg E2E Delay Sec: 3.2340678975580706


Sink speed: 25.0 m/round = 0.50 m/s
Seed = 6, Mode = random results:
FND: 164, HND: 219, LND: 21315
PDR: 0.9287840251095838, Total Generated: 18480, Total Delivered: 17163
Avg E2E Delay Rounds: 0.06356697547048884, Avg E2E Delay Sec: 3.178348773524442


Sink speed: 25.0 m/round = 0.50 m/s
Seed = 7, Mode = random results:
FND: 163, HND: 209, LND: 2946
PDR: 0.9319538601639178, Total Generated: 19767, Total Delivered: 18421
Avg E2E Delay Rounds: 0.07768307909451169, Avg E2E Delay Sec: 3.884153954725585


Sink speed: 25.0 m/round = 0.50 m/s
Seed = 8, Mode = random results:
FND: 160, HND: 204, LND: 11817
PDR: 0.9437773538323421, Total Generated: 17823, Total Delivered: 16820
Avg E2E Delay Rounds: 0.06319857312722948, Avg E2E Delay Sec: 3.1599286563614744


Sink speed: 25.0 m/round = 0.50 m/s
Seed = 9, Mode = random results:
FND: 162, HND: 228, LND: 3512
PDR: 0.9295698342182251, Total Generated: 18459, Total Delivered: 17158
Avg E2E Delay Rounds: 0.06207017134864203, Avg E2E Delay Sec: 3.1035085674321015


Sink speed: 25.0 m/round = 0.50 m/s
Seed = 10, Mode = random results:
FND: 167, HND: 205, LND: 15932
PDR: 0.9240247511433952, Total Generated: 18586, Total Delivered: 17173
Avg E2E Delay Rounds: 0.08845280381994992, Avg E2E Delay Sec: 4.422640190997496


Sink speed: 25.0 m/round = 0.50 m/s
Seed = 11, Mode = random results:
FND: 137, HND: 228, LND: 4266
PDR: 0.927924548523659, Total Generated: 18663, Total Delivered: 17316
Avg E2E Delay Rounds: 0.0807923307923308, Avg E2E Delay Sec: 4.0396165396165395


Sink speed: 25.0 m/round = 0.50 m/s
Seed = 12, Mode = random results:
FND: 143, HND: 210, LND: 14362
PDR: 0.9501699406647849, Total Generated: 17360, Total Delivered: 16494
Avg E2E Delay Rounds: 0.05850612343882624, Avg E2E Delay Sec: 2.925306171941312


Sink speed: 25.0 m/round = 0.50 m/s
Seed = 13, Mode = random results:
FND: 163, HND: 216, LND: 3844
PDR: 0.927360774818402, Total Generated: 18586, Total Delivered: 17235
Avg E2E Delay Rounds: 0.0714824485059472, Avg E2E Delay Sec: 3.57412242529736


Sink speed: 25.0 m/round = 0.50 m/s
Seed = 14, Mode = random results:
FND: 166, HND: 198, LND: 19320
PDR: 0.9131996242886348, Total Generated: 18100, Total Delivered: 16528
Avg E2E Delay Rounds: 0.03140125847047435, Avg E2E Delay Sec: 1.5700629235237173


Sink speed: 25.0 m/round = 0.50 m/s
Seed = 15, Mode = random results:
FND: 162, HND: 216, LND: 11068
PDR: 0.9208394283785304, Total Generated: 17775, Total Delivered: 16367
Avg E2E Delay Rounds: 0.08926498441986926, Avg E2E Delay Sec: 4.463249220993463


Sink speed: 25.0 m/round = 0.50 m/s
Seed = 16, Mode = random results:
FND: 148, HND: 208, LND: 16534
PDR: 0.9206632795488806, Total Generated: 17912, Total Delivered: 16490
Avg E2E Delay Rounds: 0.06901152213462705, Avg E2E Delay Sec: 3.4505761067313525


Sink speed: 25.0 m/round = 0.50 m/s
Seed = 17, Mode = random results:
FND: 135, HND: 235, LND: 4574
PDR: 0.9425983436853002, Total Generated: 19321, Total Delivered: 18211
Avg E2E Delay Rounds: 0.05485695458788644, Avg E2E Delay Sec: 2.742847729394322


Sink speed: 25.0 m/round = 0.50 m/s
Seed = 18, Mode = random results:
FND: 174, HND: 184, LND: 15025
PDR: 0.9237559707358365, Total Generated: 16540, Total Delivered: 15278
Avg E2E Delay Rounds: 0.07422437491818301, Avg E2E Delay Sec: 3.7112187459091506


Sink speed: 25.0 m/round = 0.50 m/s
Seed = 19, Mode = random results:
FND: 157, HND: 221, LND: 22818
PDR: 0.8973072149371324, Total Generated: 18532, Total Delivered: 16628
Avg E2E Delay Rounds: 0.035722877074813565, Avg E2E Delay Sec: 1.7861438537406782


Sink speed: 25.0 m/round = 0.50 m/s
Seed = 20, Mode = random results:
FND: 158, HND: 219, LND: 5134
PDR: 0.9378409029540422, Total Generated: 18518, Total Delivered: 17366
Avg E2E Delay Rounds: 0.061153979039502475, Avg E2E Delay Sec: 3.057698951975124


Sink speed: 25.0 m/round = 0.50 m/s
Seed = 21, Mode = random results:
FND: 185, HND: 226, LND: 5334
PDR: 0.9297028621367708, Total Generated: 18309, Total Delivered: 17021
Avg E2E Delay Rounds: 0.06979613418718054, Avg E2E Delay Sec: 3.4898067093590273


Sink speed: 25.0 m/round = 0.50 m/s
Seed = 22, Mode = random results:
FND: 146, HND: 239, LND: 17892
PDR: 0.9392008639308855, Total Generated: 18521, Total Delivered: 17394
Avg E2E Delay Rounds: 0.07013912843509255, Avg E2E Delay Sec: 3.506956421754628


Sink speed: 25.0 m/round = 0.50 m/s
Seed = 23, Mode = random results:
FND: 144, HND: 231, LND: 9671
PDR: 0.9192822620965502, Total Generated: 18002, Total Delivered: 16548
Avg E2E Delay Rounds: 0.07565868987188784, Avg E2E Delay Sec: 3.782934493594392


Sink speed: 25.0 m/round = 0.50 m/s
Seed = 24, Mode = random results:
FND: 168, HND: 214, LND: 17392
PDR: 0.9446389143195648, Total Generated: 18570, Total Delivered: 17541
Avg E2E Delay Rounds: 0.061456017330824925, Avg E2E Delay Sec: 3.0728008665412463


Sink speed: 25.0 m/round = 0.50 m/s
Seed = 25, Mode = random results:
FND: 143, HND: 193, LND: 10737
PDR: 0.9179322758465519, Total Generated: 17779, Total Delivered: 16319
Avg E2E Delay Rounds: 0.07770083951222502, Avg E2E Delay Sec: 3.885041975611251


Sink speed: 25.0 m/round = 0.50 m/s
Seed = 26, Mode = random results:
FND: 161, HND: 210, LND: 4356
PDR: 0.933853459972863, Total Generated: 17689, Total Delivered: 16518
Avg E2E Delay Rounds: 0.057331396052790896, Avg E2E Delay Sec: 2.8665698026395448


Sink speed: 25.0 m/round = 0.50 m/s
Seed = 27, Mode = random results:
FND: 142, HND: 252, LND: 3088
PDR: 0.9170685178315935, Total Generated: 18872, Total Delivered: 17306
Avg E2E Delay Rounds: 0.06691320929157518, Avg E2E Delay Sec: 3.345660464578759


Sink speed: 25.0 m/round = 0.50 m/s
Seed = 28, Mode = random results:
FND: 177, HND: 220, LND: 11217
PDR: 0.9445506692160612, Total Generated: 19875, Total Delivered: 18772
Avg E2E Delay Rounds: 0.05556147453654379, Avg E2E Delay Sec: 2.7780737268271896


Sink speed: 25.0 m/round = 0.50 m/s
Seed = 29, Mode = random results:
FND: 138, HND: 210, LND: 10905
PDR: 0.9153974366342893, Total Generated: 17400, Total Delivered: 15927
Avg E2E Delay Rounds: 0.09367740315188046, Avg E2E Delay Sec: 4.683870157594023


Mean DRL FND: 159.93 ± 15.62
Mean DRL HND: 223.13 ± 14.68
Mean DRL LND: 15103.50 ± 7409.88
Mean DRL PDR: 0.93 ± 0.02
Mean DRL TotalGenerated: 18563.37 ± 976.39
Mean DRL TotalDelivered: 17241.93 ± 727.60
Mean DRL Avg_E2E_Delay_Rounds: 0.06 ± 0.02
Mean DRL Avg_E2E_Delay_Sec: 3.02 ± 0.93




Mean Baseline FND: 158.67 ± 12.57
Mean Baseline HND: 217.50 ± 14.58
Mean Baseline LND: 10959.37 ± 6322.39
Mean Baseline PDR: 0.93 ± 0.01
Mean Baseline TotalGenerated: 18312.10 ± 765.88
Mean Baseline TotalDelivered: 16973.70 ± 732.16
Mean Baseline Avg_E2E_Delay_Rounds: 0.07 ± 0.01
Mean Baseline Avg_E2E_Delay_Sec: 3.31 ± 0.73
```

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(20, 20))

# EC
plt.subplot(4, 4, 1)
plot_time_series(plot_drl, 'EC', "Voronoi-RL", color='green')
plot_time_series(plot_base, 'EC', "Random", color='purple')
plt.xlabel('Round')
plt.ylabel('Coverage (EC)')
plt.title('Time')
plt.grid(True)
plt.legend()

# avg_RE
plt.subplot(4, 4, 2)
plot_time_series(plot_drl, 'avg_RE', "Voronoi-RL", color='green')
plot_time_series(plot_base, 'avg_RE', "Random", color='purple')
plt.xlabel('Round')
plt.ylabel('Load Balance (avg_RE)')
plt.title('Time')
plt.grid(True)
plt.legend()

# TH
plt.subplot(4, 4, 3)
plot_time_series(plot_drl, 'TH', "Voronoi-RL", color='green')
plot_time_series(plot_base, 'TH', "Random", color='purple')
plt.xlabel('Round')
plt.ylabel('TH')
plt.title('TH')
plt.grid(True)
plt.legend()

# TH_pps
plt.subplot(4, 4, 4)
plot_time_series(plot_drl, 'TH_pps', "Voronoi-RL", color='green')
plot_time_series(plot_base, 'TH_pps', "Random", color='purple')
plt.xlabel('Round')
plt.ylabel('THpps')
plt.title('TH_pps')
plt.grid(True)
plt.legend()

# PLR
plt.subplot(4, 4, 5)
plot_time_series(plot_drl, 'PLR', "Voronoi-RL", color='green')
plot_time_series(plot_base, 'PLR', "Random", color='purple')
plt.xlabel('Round')
plt.ylabel('PLR')
plt.title('PLR')
plt.grid(True)
plt.legend()

# LB
plt.subplot(4, 4, 6)
plot_time_series(plot_drl, 'LB', "Voronoi-RL", color='green')
plot_time_series(plot_base, 'LB', "Random", color='purple')
plt.xlabel('Round')
plt.ylabel('LB')
plt.title('LB')
plt.grid(True)
plt.legend()

# FI
plt.subplot(4, 4, 7)
plot_time_series(plot_drl, 'FI', "Voronoi-RL", color='green')
plot_time_series(plot_base, 'FI', "Random", color='purple')
plt.xlabel('Round')
plt.ylabel('FI')
plt.title('FI')
plt.grid(True)
plt.legend()

# CE
plt.subplot(4, 4, 8)
plot_time_series(plot_drl, 'CE', "Voronoi-RL", color='green')
plot_time_series(plot_base, 'CE', "Random", color='purple')
plt.xlabel('Round')
plt.ylabel('CE')
plt.title('CE')
plt.grid(True)
plt.legend()

# Buffer_Overflow_Rate
plt.subplot(4, 4, 9)
plot_time_series(plot_drl, 'Buffer_Overflow_Rate', "Voronoi-RL", color='green')
plot_time_series(plot_base, 'Buffer_Overflow_Rate', "Random", color='purple')
plt.xlabel('Round')
plt.ylabel('Bufer_Overflow_Rate')
plt.title('Buffer_Overflow_Rate')
plt.grid(True)
plt.legend()

# CA
plt.subplot(4, 4, 10)
plot_time_series(plot_drl, 'CA', "Voronoi-RL", color='green')
plot_time_series(plot_base, 'CA', "Random", color='purple')
plt.xlabel('Round')
plt.ylabel('CA')
plt.title('CA')
plt.grid(True)
plt.legend()

# RL
plt.subplot(4, 4, 11)
plot_time_series(plot_drl, 'RL', "Voronoi-RL", color='green')
plot_time_series(plot_base, 'RL', "Random", color='purple')
plt.xlabel('Round')
plt.ylabel('RL')
plt.title('RL')
plt.grid(True)
plt.legend()

# EE
plt.subplot(4, 4, 12)
plot_time_series(plot_drl, 'EE', "Voronoi-RL", color='green')
plot_time_series(plot_base, 'EE', "Random", color='purple')
plt.xlabel('Round')
plt.ylabel('EE')
plt.title('EE')
plt.grid(True)
plt.legend()

# EE_Js
plt.subplot(4, 4, 13)
plot_time_series(plot_drl, 'EE_Js', "Voronoi-RL", color='green')
plot_time_series(plot_base, 'EE_Js', "Random", color='purple')
plt.xlabel('Round')
plt.ylabel('EEJs')
plt.title('EE_Js')
plt.grid(True)
plt.legend()

# routing_Overhead_Bytes
plt.subplot(4, 4, 14)
plot_time_series(plot_drl, 'routing_Overhead_Bytes',
                 "Voronoi-RL", color='green')
plot_time_series(plot_base, 'routing_Overhead_Bytes', "Random", color='purple')
plt.xlabel('Round')
plt.ylabel('roting_Overhead_Bytes')
plt.title('routing_Overhead_Bytes')
plt.grid(True)
plt.legend()

# Traffic_Load_Pct
plt.subplot(4, 4, 15)
plot_time_series(plot_drl, 'Traffic_Load_Pct', "Voronoi-RL", color='green')
plot_time_series(plot_base, 'Traffic_Load_Pct', "Random", color='purple')
plt.xlabel('Round')
plt.ylabel('Trffic_Load_Pct')
plt.title('Traffic_Load_Pct')
plt.grid(True)
plt.legend()

# Overhead_Normalized
plt.subplot(4, 4, 16)
plot_time_series(plot_drl, 'Overhead_Normalized', "Voronoi-RL", color='green')
plot_time_series(plot_base, 'Overhead_Normalized', "Random", color='purple')
plt.xlabel('Round')
plt.ylabel('Overhead_Normalized')
plt.title('Overhead_Normalized')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.savefig('metrics_over_time.png', dpi=300, bbox_inches='tight')
plt.show()

### (200, 200) area size, 100 nodes

In [ ]:
# Run both methods
drl_lifetimes, plot_drl = run_multiple_simulations(area_size=(
    200, 200), n_nodes=100, sink_mode="eeosp", routing_mode="multi-hop", mode="DRL", n_runs=30)
baseline_lifetimes, plot_base = run_multiple_simulations(area_size=(
    200, 200), n_nodes=100, sink_mode="eeosp", routing_mode="multi-hop", mode="random", n_runs=30)


print(
    f"Mean DRL FND: {drl_lifetimes[0].mean():.2f} ± {drl_lifetimes[0].std():.2f}")
print(
    f"Mean DRL HND: {drl_lifetimes[1].mean():.2f} ± {drl_lifetimes[1].std():.2f}")
print(
    f"Mean DRL LND: {drl_lifetimes[2].mean():.2f} ± {drl_lifetimes[2].std():.2f}")
print(
    f"Mean DRL PDR: {drl_lifetimes[3].mean():.2f} ± {drl_lifetimes[3].std():.2f}")
print(
    f"Mean DRL TotalGenerated: {drl_lifetimes[4].mean():.2f} ± {drl_lifetimes[4].std():.2f}")
print(
    f"Mean DRL TotalDelivered: {drl_lifetimes[5].mean():.2f} ± {drl_lifetimes[5].std():.2f}")
print(
    f"Mean DRL Avg_E2E_Delay_Rounds: {drl_lifetimes[6].mean():.2f} ± {drl_lifetimes[6].std():.2f}")
print(
    f"Mean DRL Avg_E2E_Delay_Sec: {drl_lifetimes[7].mean():.2f} ± {drl_lifetimes[7].std():.2f}")


print("\n\n\n")
print(
    f"Mean Baseline FND: {baseline_lifetimes[0].mean():.2f} ± {baseline_lifetimes[0].std():.2f}")
print(
    f"Mean Baseline HND: {baseline_lifetimes[1].mean():.2f} ± {baseline_lifetimes[1].std():.2f}")
print(
    f"Mean Baseline LND: {baseline_lifetimes[2].mean():.2f} ± {baseline_lifetimes[2].std():.2f}")
print(
    f"Mean Baseline PDR: {baseline_lifetimes[3].mean():.2f} ± {baseline_lifetimes[3].std():.2f}")
print(
    f"Mean Baseline TotalGenerated: {baseline_lifetimes[4].mean():.2f} ± {baseline_lifetimes[4].std():.2f}")
print(
    f"Mean Baseline TotalDelivered: {baseline_lifetimes[5].mean():.2f} ± {baseline_lifetimes[5].std():.2f}")
print(
    f"Mean Baseline Avg_E2E_Delay_Rounds: {baseline_lifetimes[6].mean():.2f} ± {baseline_lifetimes[6].std():.2f}")
print(
    f"Mean Baseline Avg_E2E_Delay_Sec: {baseline_lifetimes[7].mean():.2f} ± {baseline_lifetimes[7].std():.2f}")

In [ ]:
plt.figure(figsize=(15, 5))

# CA
plt.subplot(1, 3, 1)
plot_time_series(plot_drl, 'CA', "Voronoi-RL", color='green')
plot_time_series(plot_base, 'CA', "Random", color='purple')
plt.xlabel('Round')
plt.ylabel('Coverage (CA)')
plt.title('Spatial Coverage over Time')
plt.grid(True)
plt.legend()

# LB
plt.subplot(1, 3, 2)
plot_time_series(plot_drl, 'LB', "Voronoi-RL", color='green')
plot_time_series(plot_base, 'LB', "Random", color='purple')
plt.xlabel('Round')
plt.ylabel('Load Balance (LB)')
plt.title('Load Balance over Time')
plt.grid(True)
plt.legend()

# CE
plt.subplot(1, 3, 3)
plot_time_series(plot_drl, 'EC', "Voronoi-RL", color='green')
plot_time_series(plot_base, 'EC', "Random", color='purple')
plt.xlabel('Round')
plt.ylabel('Coverage Efficiency (EC)')
plt.title('Coverage per Energy Unit (EC)')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.savefig('metrics_over_time.png', dpi=300, bbox_inches='tight')
plt.show()

### (500, 500) area size, 100 nodes

In [ ]:
# Run both methods
drl_lifetimes, plot_drl = run_multiple_simulations(area_size=(
    500, 500), n_nodes=100, sink_mode="eeosp", routing_mode="multi-hop", mode="DRL", n_runs=30, edge_threshold=0.01, tune_edge_iterations=10)
baseline_lifetimes, plot_base = run_multiple_simulations(area_size=(
    500, 500), n_nodes=100, sink_mode="eeosp", routing_mode="multi-hop", mode="random", n_runs=30)


print(
    f"Mean DRL FND: {drl_lifetimes[0].mean():.2f} ± {drl_lifetimes[0].std():.2f}")
print(
    f"Mean DRL HND: {drl_lifetimes[1].mean():.2f} ± {drl_lifetimes[1].std():.2f}")
print(
    f"Mean DRL LND: {drl_lifetimes[2].mean():.2f} ± {drl_lifetimes[2].std():.2f}")
print(
    f"Mean DRL PDR: {drl_lifetimes[3].mean():.2f} ± {drl_lifetimes[3].std():.2f}")
print(
    f"Mean DRL TotalGenerated: {drl_lifetimes[4].mean():.2f} ± {drl_lifetimes[4].std():.2f}")
print(
    f"Mean DRL TotalDelivered: {drl_lifetimes[5].mean():.2f} ± {drl_lifetimes[5].std():.2f}")
print(
    f"Mean DRL Avg_E2E_Delay_Rounds: {drl_lifetimes[6].mean():.2f} ± {drl_lifetimes[6].std():.2f}")
print(
    f"Mean DRL Avg_E2E_Delay_Sec: {drl_lifetimes[7].mean():.2f} ± {drl_lifetimes[7].std():.2f}")


print("\n\n\n")
print(
    f"Mean Baseline FND: {baseline_lifetimes[0].mean():.2f} ± {baseline_lifetimes[0].std():.2f}")
print(
    f"Mean Baseline HND: {baseline_lifetimes[1].mean():.2f} ± {baseline_lifetimes[1].std():.2f}")
print(
    f"Mean Baseline LND: {baseline_lifetimes[2].mean():.2f} ± {baseline_lifetimes[2].std():.2f}")
print(
    f"Mean Baseline PDR: {baseline_lifetimes[3].mean():.2f} ± {baseline_lifetimes[3].std():.2f}")
print(
    f"Mean Baseline TotalGenerated: {baseline_lifetimes[4].mean():.2f} ± {baseline_lifetimes[4].std():.2f}")
print(
    f"Mean Baseline TotalDelivered: {baseline_lifetimes[5].mean():.2f} ± {baseline_lifetimes[5].std():.2f}")
print(
    f"Mean Baseline Avg_E2E_Delay_Rounds: {baseline_lifetimes[6].mean():.2f} ± {baseline_lifetimes[6].std():.2f}")
print(
    f"Mean Baseline Avg_E2E_Delay_Sec: {baseline_lifetimes[7].mean():.2f} ± {baseline_lifetimes[7].std():.2f}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(15, 5))

# CA
plt.subplot(1, 3, 1)
plot_time_series(plot_drl, 'CA', "Voronoi-RL", color='green')
plot_time_series(plot_base, 'CA', "Random", color='purple')
plt.xlabel('Round')
plt.ylabel('Coverage (CA)')
plt.title('Spatial Coverage over Time')
plt.grid(True)
plt.legend()

# LB
plt.subplot(1, 3, 2)
plot_time_series(plot_drl, 'LB', "Voronoi-RL", color='green')
plot_time_series(plot_base, 'LB', "Random", color='purple')
plt.xlabel('Round')
plt.ylabel('Load Balance (LB)')
plt.title('Load Balance over Time')
plt.grid(True)
plt.legend()

# CE
plt.subplot(1, 3, 3)
plot_time_series(plot_drl, 'EC', "Voronoi-RL", color='green')
plot_time_series(plot_base, 'EC', "Random", color='purple')
plt.xlabel('Round')
plt.ylabel('Coverage Efficiency (EC)')
plt.title('Coverage per Energy Unit (EC)')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.savefig('metrics_over_time.png', dpi=300, bbox_inches='tight')
plt.show()

### (1000, 1000) area size, 100 nodes

In [ ]:
# Run both methods
drl_lifetimes, plot_drl = run_multiple_simulations(area_size=(
    1000, 1000), n_nodes=100, sink_mode="eeosp", routing_mode="multi-hop", mode="DRL", n_runs=30, edge_threshold=0.01, tune_edge_iterations=10)
baseline_lifetimes, plot_base = run_multiple_simulations(area_size=(
    1000, 1000), n_nodes=100, sink_mode="eeosp", routing_mode="multi-hop", mode="random", n_runs=30)


print(
    f"Mean DRL FND: {drl_lifetimes[0].mean():.2f} ± {drl_lifetimes[0].std():.2f}")
print(
    f"Mean DRL HND: {drl_lifetimes[1].mean():.2f} ± {drl_lifetimes[1].std():.2f}")
print(
    f"Mean DRL LND: {drl_lifetimes[2].mean():.2f} ± {drl_lifetimes[2].std():.2f}")
print(
    f"Mean DRL PDR: {drl_lifetimes[3].mean():.2f} ± {drl_lifetimes[3].std():.2f}")
print(
    f"Mean DRL TotalGenerated: {drl_lifetimes[4].mean():.2f} ± {drl_lifetimes[4].std():.2f}")
print(
    f"Mean DRL TotalDelivered: {drl_lifetimes[5].mean():.2f} ± {drl_lifetimes[5].std():.2f}")
print(
    f"Mean DRL Avg_E2E_Delay_Rounds: {drl_lifetimes[6].mean():.2f} ± {drl_lifetimes[6].std():.2f}")
print(
    f"Mean DRL Avg_E2E_Delay_Sec: {drl_lifetimes[7].mean():.2f} ± {drl_lifetimes[7].std():.2f}")


print("\n\n\n")
print(
    f"Mean Baseline FND: {baseline_lifetimes[0].mean():.2f} ± {baseline_lifetimes[0].std():.2f}")
print(
    f"Mean Baseline HND: {baseline_lifetimes[1].mean():.2f} ± {baseline_lifetimes[1].std():.2f}")
print(
    f"Mean Baseline LND: {baseline_lifetimes[2].mean():.2f} ± {baseline_lifetimes[2].std():.2f}")
print(
    f"Mean Baseline PDR: {baseline_lifetimes[3].mean():.2f} ± {baseline_lifetimes[3].std():.2f}")
print(
    f"Mean Baseline TotalGenerated: {baseline_lifetimes[4].mean():.2f} ± {baseline_lifetimes[4].std():.2f}")
print(
    f"Mean Baseline TotalDelivered: {baseline_lifetimes[5].mean():.2f} ± {baseline_lifetimes[5].std():.2f}")
print(
    f"Mean Baseline Avg_E2E_Delay_Rounds: {baseline_lifetimes[6].mean():.2f} ± {baseline_lifetimes[6].std():.2f}")
print(
    f"Mean Baseline Avg_E2E_Delay_Sec: {baseline_lifetimes[7].mean():.2f} ± {baseline_lifetimes[7].std():.2f}")

In [ ]:
plt.figure(figsize=(15, 5))

# CA
plt.subplot(1, 3, 1)
plot_time_series(plot_drl, 'CA', "Voronoi-RL", color='green')
plot_time_series(plot_base, 'CA', "Random", color='purple')
plt.xlabel('Round')
plt.ylabel('Coverage (CA)')
plt.title('Spatial Coverage over Time')
plt.grid(True)
plt.legend()

# LB
plt.subplot(1, 3, 2)
plot_time_series(plot_drl, 'LB', "Voronoi-RL", color='green')
plot_time_series(plot_base, 'LB', "Random", color='purple')
plt.xlabel('Round')
plt.ylabel('Load Balance (LB)')
plt.title('Load Balance over Time')
plt.grid(True)
plt.legend()

# CE
plt.subplot(1, 3, 3)
plot_time_series(plot_drl, 'EC', "Voronoi-RL", color='green')
plot_time_series(plot_base, 'EC', "Random", color='purple')
plt.xlabel('Round')
plt.ylabel('Coverage Efficiency (EC)')
plt.title('Coverage per Energy Unit (EC)')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.savefig('metrics_over_time.png', dpi=300, bbox_inches='tight')
plt.show()

## Different MS patterns

In [ ]:
# Run both methods
eeosp_lifetimes, plot_eeosp = run_multiple_simulations(area_size=(
    100, 100), n_nodes=100, sink_mode="eeosp", routing_mode="multi-hop", mode="random", n_runs=30)
adaptive_lifetimes, plot_adaptive = run_multiple_simulations(area_size=(
    100, 100), n_nodes=100, sink_mode="adaptive", routing_mode="multi-hop", mode="random", n_runs=30)
fixed_lifetimes, plot_fixed = run_multiple_simulations(area_size=(
    100, 100), n_nodes=100, sink_mode="fixed", routing_mode="multi-hop", mode="random", n_runs=30)
random_lifetimes, plot_random = run_multiple_simulations(area_size=(
    100, 100), n_nodes=100, sink_mode="random", routing_mode="multi-hop", mode="random", n_runs=30)

print(
    f"Mean eeosp FND: {eeosp_lifetimes[0].mean():.2f} ± {eeosp_lifetimes[0].std():.2f}")
print(
    f"Mean eeosp HND: {eeosp_lifetimes[1].mean():.2f} ± {eeosp_lifetimes[1].std():.2f}")
print(
    f"Mean eeosp LND: {eeosp_lifetimes[2].mean():.2f} ± {eeosp_lifetimes[2].std():.2f}")
print(
    f"Mean eeosp PDR: {eeosp_lifetimes[3].mean():.2f} ± {eeosp_lifetimes[3].std():.2f}")
print(
    f"Mean eeosp TotalGenerated: {eeosp_lifetimes[4].mean():.2f} ± {eeosp_lifetimes[4].std():.2f}")
print(
    f"Mean eeosp TotalDelivered: {eeosp_lifetimes[5].mean():.2f} ± {eeosp_lifetimes[5].std():.2f}")
print(
    f"Mean eeosp Avg_E2E_Delay_Rounds: {eeosp_lifetimes[6].mean():.2f} ± {eeosp_lifetimes[6].std():.2f}")
print(
    f"Mean eeosp Avg_E2E_Delay_Sec: {eeosp_lifetimes[7].mean():.2f} ± {eeosp_lifetimes[7].std():.2f}")


print("\n\n")
print(
    f"Mean adaptive FND: {adaptive_lifetimes[0].mean():.2f} ± {adaptive_lifetimes[0].std():.2f}")
print(
    f"Mean adaptive HND: {adaptive_lifetimes[1].mean():.2f} ± {adaptive_lifetimes[1].std():.2f}")
print(
    f"Mean adaptive LND: {adaptive_lifetimes[2].mean():.2f} ± {adaptive_lifetimes[2].std():.2f}")
print(
    f"Mean adaptive PDR: {adaptive_lifetimes[3].mean():.2f} ± {adaptive_lifetimes[3].std():.2f}")
print(
    f"Mean adaptive TotalGenerated: {adaptive_lifetimes[4].mean():.2f} ± {adaptive_lifetimes[4].std():.2f}")
print(
    f"Mean adaptive TotalDelivered: {adaptive_lifetimes[5].mean():.2f} ± {adaptive_lifetimes[5].std():.2f}")
print(
    f"Mean adaptive Avg_E2E_Delay_Rounds: {adaptive_lifetimes[6].mean():.2f} ± {adaptive_lifetimes[6].std():.2f}")
print(
    f"Mean adaptive Avg_E2E_Delay_Sec: {adaptive_lifetimes[7].mean():.2f} ± {adaptive_lifetimes[7].std():.2f}")


print("\n\n")
print(
    f"Mean fixed FND: {fixed_lifetimes[0].mean():.2f} ± {fixed_lifetimes[0].std():.2f}")
print(
    f"Mean fixed HND: {fixed_lifetimes[1].mean():.2f} ± {fixed_lifetimes[1].std():.2f}")
print(
    f"Mean fixed LND: {fixed_lifetimes[2].mean():.2f} ± {fixed_lifetimes[2].std():.2f}")
print(
    f"Mean fixed PDR: {fixed_lifetimes[3].mean():.2f} ± {fixed_lifetimes[3].std():.2f}")
print(
    f"Mean fixed TotalGenerated: {fixed_lifetimes[4].mean():.2f} ± {fixed_lifetimes[4].std():.2f}")
print(
    f"Mean fixed TotalDelivered: {fixed_lifetimes[5].mean():.2f} ± {fixed_lifetimes[5].std():.2f}")
print(
    f"Mean fixed Avg_E2E_Delay_Rounds: {fixed_lifetimes[6].mean():.2f} ± {fixed_lifetimes[6].std():.2f}")
print(
    f"Mean fixed Avg_E2E_Delay_Sec: {fixed_lifetimes[7].mean():.2f} ± {fixed_lifetimes[7].std():.2f}")


print("\n\n")
print(
    f"Mean random FND: {random_lifetimes[0].mean():.2f} ± {random_lifetimes[0].std():.2f}")
print(
    f"Mean random HND: {random_lifetimes[1].mean():.2f} ± {random_lifetimes[1].std():.2f}")
print(
    f"Mean random LND: {random_lifetimes[2].mean():.2f} ± {random_lifetimes[2].std():.2f}")
print(
    f"Mean random PDR: {random_lifetimes[3].mean():.2f} ± {random_lifetimes[3].std():.2f}")
print(
    f"Mean random TotalGenerated: {random_lifetimes[4].mean():.2f} ± {random_lifetimes[4].std():.2f}")
print(
    f"Mean random TotalDelivered: {random_lifetimes[5].mean():.2f} ± {random_lifetimes[5].std():.2f}")
print(
    f"Mean random Avg_E2E_Delay_Rounds: {random_lifetimes[6].mean():.2f} ± {random_lifetimes[6].std():.2f}")
print(
    f"Mean random Avg_E2E_Delay_Sec: {random_lifetimes[7].mean():.2f} ± {random_lifetimes[7].std():.2f}")

In [ ]:
plt.figure(figsize=(15, 5))

# CA
plt.subplot(1, 3, 1)
plot_time_series(plot_eeosp, 'CA', "EEOSP", color='green')
plot_time_series(plot_adaptive, 'CA', "adaptive", color='purple')
plot_time_series(plot_fixed, 'CA', "fixed", color='blue')
plot_time_series(plot_random, 'CA', "random", color='red')
plt.xlabel('Round')
plt.ylabel('Coverage (CA)')
plt.title('Spatial Coverage over Time')
plt.grid(True)
plt.legend()

# LB
plt.subplot(1, 3, 2)
plot_time_series(plot_eeosp, 'LB', "EEOSP", color='green')
plot_time_series(plot_adaptive, 'LB', "adaptive", color='purple')
plot_time_series(plot_fixed, 'LB', "fixed", color='blue')
plot_time_series(plot_random, 'LB', "random", color='red')
plt.xlabel('Round')
plt.ylabel('Load Balance (LB)')
plt.title('Load Balance over Time')
plt.grid(True)
plt.legend()

# CE
plt.subplot(1, 3, 3)
plot_time_series(plot_eeosp, 'EC', "EEOSP", color='green')
plot_time_series(plot_adaptive, 'EC', "adaptive", color='purple')
plot_time_series(plot_fixed, 'EC', "fixed", color='blue')
plot_time_series(plot_random, 'EC', "random", color='red')
plt.xlabel('Round')
plt.ylabel('Coverage Efficiency (EC)')
plt.title('Coverage per Energy Unit (EC)')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.savefig('metrics_over_time.png', dpi=300, bbox_inches='tight')
plt.show()

## Different clustering methods

In [ ]:
# Run both methods
OGSA_lifetimes, plot_OGSA = run_multiple_simulations(area_size=(
    100, 100), n_nodes=100, sink_mode="eeosp", routing_mode="multi-hop", mode="random", n_runs=30, CHS="optimizer")
adaptive_lifetimes, plot_adaptive = run_multiple_simulations(area_size=(
    100, 100), n_nodes=100, sink_mode="eeosp", routing_mode="multi-hop", mode="random", n_runs=30, CHS="adaptive")
random_lifetimes, plot_random = run_multiple_simulations(area_size=(
    100, 100), n_nodes=100, sink_mode="eeosp", routing_mode="multi-hop", mode="random", n_runs=30, CHS="random")


print(
    f"Mean DRL FND: {OGSA_lifetimes[0].mean():.2f} ± {OGSA_lifetimes[0].std():.2f}")
print(
    f"Mean OGSA HND: {OGSA_lifetimes[1].mean():.2f} ± {OGSA_lifetimes[1].std():.2f}")
print(
    f"Mean OGSA LND: {OGSA_lifetimes[2].mean():.2f} ± {OGSA_lifetimes[2].std():.2f}")
print(
    f"Mean OGSA PDR: {OGSA_lifetimes[3].mean():.2f} ± {OGSA_lifetimes[3].std():.2f}")
print(
    f"Mean OGSA TotalGenerated: {OGSA_lifetimes[4].mean():.2f} ± {OGSA_lifetimes[4].std():.2f}")
print(
    f"Mean OGSA TotalDelivered: {OGSA_lifetimes[5].mean():.2f} ± {OGSA_lifetimes[5].std():.2f}")
print(
    f"Mean OGSA Avg_E2E_Delay_Rounds: {OGSA_lifetimes[6].mean():.2f} ± {OGSA_lifetimes[6].std():.2f}")
print(
    f"Mean OGSA Avg_E2E_Delay_Sec: {OGSA_lifetimes[7].mean():.2f} ± {OGSA_lifetimes[7].std():.2f}")


print("\n\n\n")
print(
    f"Mean adaptive FND: {adaptive_lifetimes[0].mean():.2f} ± {adaptive_lifetimes[0].std():.2f}")
print(
    f"Mean adaptive HND: {adaptive_lifetimes[1].mean():.2f} ± {adaptive_lifetimes[1].std():.2f}")
print(
    f"Mean adaptive LND: {adaptive_lifetimes[2].mean():.2f} ± {adaptive_lifetimes[2].std():.2f}")
print(
    f"Mean adaptive PDR: {adaptive_lifetimes[3].mean():.2f} ± {adaptive_lifetimes[3].std():.2f}")
print(
    f"Mean adaptive TotalGenerated: {adaptive_lifetimes[4].mean():.2f} ± {adaptive_lifetimes[4].std():.2f}")
print(
    f"Mean adaptive TotalDelivered: {adaptive_lifetimes[5].mean():.2f} ± {adaptive_lifetimes[5].std():.2f}")
print(
    f"Mean adaptive Avg_E2E_Delay_Rounds: {adaptive_lifetimes[6].mean():.2f} ± {adaptive_lifetimes[6].std():.2f}")
print(
    f"Mean adaptive Avg_E2E_Delay_Sec: {adaptive_lifetimes[7].mean():.2f} ± {adaptive_lifetimes[7].std():.2f}")


print("\n\n\n")
print(
    f"Mean random FND: {random_lifetimes[0].mean():.2f} ± {random_lifetimes[0].std():.2f}")
print(
    f"Mean random HND: {random_lifetimes[1].mean():.2f} ± {random_lifetimes[1].std():.2f}")
print(
    f"Mean random LND: {random_lifetimes[2].mean():.2f} ± {random_lifetimes[2].std():.2f}")
print(
    f"Mean random PDR: {random_lifetimes[3].mean():.2f} ± {random_lifetimes[3].std():.2f}")
print(
    f"Mean random TotalGenerated: {random_lifetimes[4].mean():.2f} ± {random_lifetimes[4].std():.2f}")
print(
    f"Mean random TotalDelivered: {random_lifetimes[5].mean():.2f} ± {random_lifetimes[5].std():.2f}")
print(
    f"Mean random Avg_E2E_Delay_Rounds: {random_lifetimes[6].mean():.2f} ± {random_lifetimes[6].std():.2f}")
print(
    f"Mean random Avg_E2E_Delay_Sec: {random_lifetimes[7].mean():.2f} ± {random_lifetimes[7].std():.2f}")

In [ ]:
plt.figure(figsize=(15, 5))

# CA
plt.subplot(1, 3, 1)
plot_time_series(plot_OGSA, 'CA', "OGSA", color='green')
plot_time_series(plot_adaptive, 'CA', "adaptive", color='purple')
plot_time_series(plot_random, 'CA', "random", color='red')
plt.xlabel('Round')
plt.ylabel('Coverage (CA)')
plt.title('Spatial Coverage over Time')
plt.grid(True)
plt.legend()

# LB
plt.subplot(1, 3, 2)
plot_time_series(plot_OGSA, 'LB', "OGSA", color='green')
plot_time_series(plot_adaptive, 'LB', "Random", color='purple')
plot_time_series(plot_random, 'LB', "random", color='red')
plt.xlabel('Round')
plt.ylabel('Load Balance (LB)')
plt.title('Load Balance over Time')
plt.grid(True)
plt.legend()

# CE
plt.subplot(1, 3, 3)
plot_time_series(plot_OGSA, 'EC', "OGSA", color='green')
plot_time_series(plot_adaptive, 'EC', "Random", color='purple')
plot_time_series(plot_random, 'EC', "random", color='red')
plt.xlabel('Round')
plt.ylabel('Coverage Efficiency (EC)')
plt.title('Coverage per Energy Unit (EC)')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.savefig('metrics_over_time.png', dpi=300, bbox_inches='tight')
plt.show()